# Bootstrap wake-word positives with voice conversion (voiceclonnx)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/nb11_voice_conversion.ipynb)

> **Turn a handful of real wake-word recordings into a large, speaker-diverse positive set — entirely pure-ONNX, no PyTorch at inference.**

A good wake-word model needs **positive** examples (clips that *do* contain your phrase) spoken by **many different people**: men, women, children, accents, pitches, speaking rates. Recording hundreds of real speakers is expensive. This notebook closes that gap with **voice conversion (VC)**.

### What is voice conversion?

**Voice conversion** takes an existing recording and re-renders the *same words* in a *different person's voice*. The words, timing and prosody come from the **source** clip; the **timbre** (the unique colour and identity of a voice) comes from a **donor** (also called a **reference** or **target** speaker). It is **audio-to-audio**: it converts speech you already have — it does **not** generate speech from text (that is TTS, a separate tool).

> **Glossary**
> - **Source** — a real recording of your wake word. Provides the *words*.
> - **Donor / reference / target** — a short clip of someone else talking (any words). Provides the *voice identity*.
> - **Timbre** — the tonal fingerprint of a voice that makes one speaker sound different from another.
> - **Zero-shot** — the engine can imitate a brand-new donor it has never seen before, from a single short clip — no per-speaker training.

So: take 10 real recordings of *"hey jarvis"*, voice-convert each into 100 different donor voices, and you have **1,000 speaker-diverse positives** — all genuinely saying your phrase.

### Why voiceclonnx?

All voice conversion in this repo is delegated to **[voiceclonnx](https://github.com/TigreGotico/voiceclonnx)** — a pure-ONNX, multi-engine voice-cloning library. **Zero PyTorch at runtime** (only `onnxruntime`, `numpy`, `soundfile`); models download from the Hugging Face Hub on first use. One `pip install` enables every engine.

---

## What you'll build

```
 real positives (few)           donor voices (many)
   hey_jarvis_01.wav      ×        speaker_A.wav
   hey_jarvis_02.wav               speaker_B.wav          voiceclonnx
        ...                            ...           ───────────────────►   vc_*.wav  (many, label 1)
                                                       (zero-shot VC)        appended to metadata.csv
```

| Cell | Step | What happens |
|------|------|--------------|
| 2 | **Install** | Install `ww_trainer[vc]` (pulls in voiceclonnx) |
| 3 | **Configure** | Source dir, donor dir, output dir, engine, target count |
| 4 | **Pick an engine** | List available engines; choose one (default `knnvc`) |
| 5 | **Run conversion** | Voice-convert source × donor pairs into new positives |
| 6 | **Listen / inspect** | Play a few converted clips inline |
| 7 | **Append to CSV** | Add the new files to a training manifest as label `1` |
| 8 | **What next** | Feed the enlarged positive set into training |

> **Scope:** voiceclonnx is **audio-to-audio only**. To synthesise positives *from text* instead, use the TTS datagen pipeline (see `notebooks/kaggle_quickstart.ipynb`). The two are complementary: TTS gives you a base set from text, VC multiplies real recordings into many speakers.

## Cell 2 — Install

Installs `ww_trainer` with the **`[vc]`** extra, which pulls in the pure-ONNX **voiceclonnx** library and its runtime dependencies (`onnxruntime`, `numpy`, `soundfile`, `huggingface_hub`). **No PyTorch is required for voice conversion.**

> **First-run model download:** voiceclonnx does not bundle weights. The first time you use an engine, its ONNX models are downloaded from the Hugging Face Hub and cached locally (typically tens to a few hundred MB, depending on engine). Subsequent runs are offline-fast. On Kaggle/Colab this happens automatically; locally it caches under `~/.cache/huggingface/`.

> **Already installed?** Re-running this cell is safe — `pip` skips packages that are already present.

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Core audio stack (already present on Kaggle/Colab; listed for local runs).
_pip("numpy", "soundfile", "onnxruntime")

# ww_trainer with the voice-conversion extra → installs voiceclonnx (pure-ONNX).
_pip("ww_trainer[vc]")

# Optional: the [datagen] extra adds TTS + HuggingFace downloads, only needed
# if you also want to generate base positives from text in this session.
# _pip("ww_trainer[datagen]")

import ww_trainer
import voiceclonnx
print(f"ww_trainer : {ww_trainer.__version__}")
print(f"voiceclonnx: import OK (engines auto-registered)")
print(f"Python     : {sys.version.split()[0]}")

## Cell 3 — Configuration

Point the notebook at three folders and pick how many positives to produce. Every value can also be supplied as an environment variable of the same name.

| Variable | Default | Meaning |
|----------|---------|---------|
| `SOURCE_DIR` | `./vc_demo/sources` | Folder of **real wake-word recordings** (`.wav`) — the *words*. |
| `DONOR_DIR` | `./vc_demo/donors` | Folder of **donor/reference voices** (`.wav`, any words) — the *voices*. |
| `OUTPUT_DIR` | `./vc_demo/converted` | Where converted positives are written. |
| `WW_VC_ENGINE` | `knnvc` | voiceclonnx engine alias (see the engine table in Cell 4). |
| `N_TARGET` | `12` | How many converted clips to produce in total. |
| `TRAIN_CSV` | `./vc_demo/metadata.csv` | Manifest the new positives are appended to (Cell 7). |

> **Donor tip:** good donors are short (~5–30 s), reasonably clean clips of *different* people speaking *anything*. The wider the variety of donors, the more speaker-diverse your positives. A folder of `not-wake-word` speech clips makes an excellent free donor pool — each one is a real human voice.

If you have no audio yet, the next cell will synthesise a tiny demo set so the notebook runs end-to-end; replace the folders with your own recordings for real use.

In [ ]:
import os
from pathlib import Path

# ── Folders ───────────────────────────────────────────────────────────────────
SOURCE_DIR = Path(os.environ.get("SOURCE_DIR", "./vc_demo/sources"))     # real positives (words)
DONOR_DIR  = Path(os.environ.get("DONOR_DIR",  "./vc_demo/donors"))      # reference voices (timbre)
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "./vc_demo/converted"))   # converted positives out
TRAIN_CSV  = Path(os.environ.get("TRAIN_CSV",  "./vc_demo/metadata.csv"))# manifest to append to

# ── Engine + amount ───────────────────────────────────────────────────────────
# WW_VC_ENGINE is the env var the whole repo uses to select a voiceclonnx engine.
VC_ENGINE  = os.environ.get("WW_VC_ENGINE", "knnvc")
N_TARGET   = int(os.environ.get("N_TARGET", "12"))   # total converted clips to produce

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
DONOR_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Demo fallback: if no audio is present, synthesise a tiny set so the ────────
# notebook is runnable out-of-the-box. Replace with real recordings for real use.
def _have_wavs(d: Path) -> bool:
    return any(d.rglob("*.wav"))

if not _have_wavs(SOURCE_DIR) or not _have_wavs(DONOR_DIR):
    import numpy as np, soundfile as sf
    print("No audio found — writing a tiny synthetic demo set (replace with real WAVs).")
    rng = np.random.default_rng(0)
    sr = 16000
    def _tone(freq, dur=1.0, noise=0.01):
        t = np.linspace(0, dur, int(sr * dur), endpoint=False)
        wav = 0.3 * np.sin(2 * np.pi * freq * t) + noise * rng.standard_normal(t.shape)
        return wav.astype(np.float32)
    if not _have_wavs(SOURCE_DIR):
        for i, f in enumerate([180, 200, 220]):   # stand-ins for "real positives"
            sf.write(SOURCE_DIR / f"source_{i:02d}.wav", _tone(f), sr)
    if not _have_wavs(DONOR_DIR):
        for i, f in enumerate([140, 260, 320, 400]):  # stand-ins for "donor voices"
            sf.write(DONOR_DIR / f"donor_{i:02d}.wav", _tone(f), sr)

sources = sorted(SOURCE_DIR.rglob("*.wav"))
donors  = sorted(DONOR_DIR.rglob("*.wav"))
print(f"Sources : {len(sources)} wav  in {SOURCE_DIR}")
print(f"Donors  : {len(donors)} wav  in {DONOR_DIR}")
print(f"Engine  : voiceclonnx:{VC_ENGINE}")
print(f"Target  : {N_TARGET} converted positives → {OUTPUT_DIR}")
assert sources and donors, "Need at least one source and one donor WAV."

## Cell 4 — Pick an engine

voiceclonnx ships several engines. They differ in **output sample rate**, **conversion style** (any-to-any vs any-to-one) and **quality/speed** trade-offs. For bootstrapping wake-word positives you want a **zero-shot any-to-any** engine so any donor clip works without per-speaker setup — `knnvc` is the recommended default.

| Engine | Output SR | One-line trade-off |
|--------|-----------|--------------------|
| **`knnvc`** *(default)* | 16 kHz | WavLM + kNN matching + HiFi-GAN. Zero-shot any-to-any; robust, simple, **matches the 16 kHz training rate**. |
| `facodec` | 16 kHz | Factorised codec (content/timbre disentangled). Zero-shot any-to-any; strong identity transfer. |
| `freevc` | 16 kHz | WavLM + GE2E speaker encoder + VITS decoder. Zero-shot any-to-any; clean, natural output. |
| `openvoice` | 22 kHz | Tone-colour reference encoder + VITS converter. Zero-shot any-to-any; expressive. |
| `rvc` | 40 kHz | ContentVec + RMVPE F0 + VITS. **Any-to-one** — donor must be a pre-trained RVC `.onnx` model, not a WAV. |
| `linacodec` | 16 kHz | Neural-codec VC; compact tokens, fast decode. Zero-shot any-to-any. |
| `chatterbox` | 24 kHz | AR codec-LM VC path. Zero-shot any-to-any; highest fidelity, slowest. |

> **Sample rate doesn't matter for training:** the ww_trainer dataset loader resamples everything to 16 kHz on load, so a 24 kHz engine is fine. `knnvc` outputs 16 kHz directly, which keeps files smallest.

> **`rvc` is special:** it is *any-to-one* — the "donor" is a trained RVC model file, not a reference WAV. Skip it for the donor-folder workflow in this notebook unless you have RVC `.onnx` models.

The cell below lists every engine actually available in your install and confirms your choice is valid.

In [ ]:
from ww_trainer.vc_helpers import list_engines

available = list_engines()
print("Available voiceclonnx engines:")
print("  " + ", ".join(available))

assert VC_ENGINE in available, (
    f"Engine {VC_ENGINE!r} not available. Choose one of: {', '.join(available)}\n"
    f"Set it via the WW_VC_ENGINE env var or VC_ENGINE in Cell 3."
)
print(f"\nSelected engine: {VC_ENGINE}  (valid)")

## Cell 5 — Run voice conversion

There are two equivalent ways to convert, both in `ww_trainer`:

1. **Batch helper** — `ww_trainer.datagen.voice_convert_batch(...)` converts every WAV in a folder, picking a random donor per file, until it reaches `n_target` (reusing sources with new donors when `n_target` exceeds the number of sources). This is the one-liner for "make me N positives".
2. **Manual loop** — load the backend once with `ww_trainer.vc_helpers.load_vc_backend(engine=...)` and call `.vc(source, donor, out)` over chosen source × donor pairs. Use this when you want explicit control over which pairs get made (e.g. every source × every donor).

Both load the engine once and reuse it — the model is only initialised a single time. The cell below uses the **batch helper** for the bulk run, then demonstrates the **manual loop** on a couple of explicit pairs so you can see the lower-level API.

> **First call is slow:** the engine's ONNX models download from Hugging Face on the very first `.vc()` of the session. After that each conversion is fast (CPU, no GPU needed).

> **Per-pair failures are non-fatal:** the batch helper logs a warning and skips a clip if a single conversion fails (e.g. a corrupt donor), so one bad file won't abort the run.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s — %(message)s")

# ── Option 1: batch helper — make N_TARGET positives in one call ───────────────
from ww_trainer.datagen import voice_convert_batch

converted = voice_convert_batch(
    input_dir=SOURCE_DIR,     # real wake-word recordings (the words)
    output_dir=OUTPUT_DIR,    # where converted clips land
    vc_refs_dir=DONOR_DIR,    # donor/reference voices (the timbre)
    n_target=N_TARGET,        # total clips to produce (sources reused with new donors)
    engine=VC_ENGINE,         # voiceclonnx engine alias
)
print(f"\nBatch helper produced {len(converted)} converted positives.")

# ── Option 2: manual loop — explicit source × donor control ────────────────────
# Load the engine once, then convert chosen pairs yourself. Equivalent under the
# hood; use this when you want every-source-×-every-donor or custom naming.
from ww_trainer.vc_helpers import load_vc_backend

vc = load_vc_backend(engine=VC_ENGINE)
print(f"Backend: {vc.name}  (output sample rate {vc.sample_rate} Hz)")

manual_dir = OUTPUT_DIR / "manual"
manual_dir.mkdir(parents=True, exist_ok=True)

manual_pairs = [(sources[0], donors[0]), (sources[0], donors[-1])]
manual_out = []
for src, donor in manual_pairs:
    out_path = manual_dir / f"vc_{src.stem}__{donor.stem}.wav"
    vc.vc(str(src), str(donor), str(out_path))   # source words, donor voice → out
    manual_out.append(out_path)
    print(f"  {src.name}  +  voice of {donor.name}  ->  {out_path.name}")

# All converted files (batch + manual) — these are the new positives.
all_converted = list(converted) + manual_out
print(f"\nTotal converted positives this run: {len(all_converted)}")

## Cell 6 — Listen and inspect

Always sanity-check a few converted clips by ear before adding hundreds to your training set. A good conversion **still clearly says your wake word** (the words must survive) but **sounds like a different person** (the timbre changed). Discard any clip where the phrase became unintelligible.

The cell plays a few outputs inline with `IPython.display.Audio` and prints each file's duration and sample rate.

> With the synthetic demo audio these will be pure tones, not speech — that is expected. Swap in real recordings to hear genuine voice conversion.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

preview = all_converted[:3]
if not preview:
    print("Nothing was converted — check the logs in Cell 5.")
for p in preview:
    wav, sr = sf.read(str(p))
    dur = len(wav) / sr
    print(f"{p.name}  —  {dur:.2f}s @ {sr} Hz")
    display(Audio(data=wav, rate=sr))

## Cell 7 — Append to a training manifest

ww_trainer reads datasets from a **`metadata.csv`** with one `path,label` row per clip:

```
/data/positives/hey_jarvis_01.wav,1
/data/negatives/some_speech.wav,0
/data/vc/vc_abc123.wav,1
```

- **`1`** = positive (contains the wake word)
- **`0`** = negative (does not)

Converted clips are all positives, so every new row gets label **`1`**. The cell appends the freshly converted files to `TRAIN_CSV`, skipping any path already present so it is safe to re-run. The same `path,label` format is parsed by `ww_trainer.utils.read_dataset_csv` (header optional).

> **Tip:** keep your VC positives in their own folder and CSV so you can ablate "with VC vs without" easily, then concatenate CSVs at training time.

In [ ]:
from ww_trainer.utils import read_dataset_csv

# Existing rows (if any) — avoid duplicate lines on re-run.
existing_paths = set()
if TRAIN_CSV.exists():
    existing_paths = {p for p, _ in read_dataset_csv(TRAIN_CSV)}

TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)
added = 0
with open(TRAIN_CSV, "a") as f:
    for p in all_converted:
        ap = str(Path(p).resolve())
        if ap in existing_paths:
            continue
        f.write(f"{ap},1\n")     # label 1 = positive (wake word present)
        existing_paths.add(ap)
        added += 1

print(f"Appended {added} positive rows to {TRAIN_CSV}")

# Show the tail of the manifest so you can see the format.
rows = read_dataset_csv(TRAIN_CSV)
print(f"\nManifest now has {len(rows)} rows. Last 5:")
for path, label in rows[-5:]:
    print(f"  {label}  {path}")

## What next

You now have a folder of voice-converted positives and a `metadata.csv` that references them. Put them to work:

### Feed them into training
- **Quickstart / full training** — point a training run at a dataset that includes these VC positives. See `notebooks/kaggle_quickstart.ipynb` for the end-to-end "dataset → train → ONNX" flow, and `notebooks/kaggle_experiments.ipynb` to compare tiers and losses.
- **Per-epoch VC during infinite training** — `notebooks/kaggle_infinite.ipynb` can synthesise fresh VC positives *every epoch* automatically: set `VC_PER_EPOCH > 0` and `WW_VC_ENGINE`. This notebook is the manual, inspectable version of that same machinery.
- **Command-line batch** — `scripts/data/generate_vc_positives.py` does this at scale, using your `not-wake-word` clips as a donor pool and updating the train/test CSVs. Select the engine with `--vc-engine` or `WW_VC_ENGINE`.

### Get more diversity
- **More donors** — the single biggest lever. Each donor is one more "speaker" in your positive set. A large `not-wake-word` speech corpus is a free, realistic donor pool.
- **More source recordings** — even 10–20 real recordings, each crossed with hundreds of donors, yields a rich set. Variety in the *sources* (different rooms, distances, intonations) helps too.
- **Try another engine** — re-run with `WW_VC_ENGINE=freevc` or `facodec` and mix outputs from several engines for extra acoustic variety.

### Remember the scope
- voiceclonnx is **audio-to-audio only** — it re-voices speech you already have; it never invents new words. To create positives **from text**, use the TTS datagen pipeline (`ww_trainer[datagen]`, shown in `kaggle_quickstart.ipynb`). The strongest datasets combine **TTS** (cheap base coverage from text) **+ VC** (real recordings multiplied across many speakers) **+ real recordings** (ground truth).

### Troubleshooting

| Symptom | Fix |
|---------|-----|
| `RuntimeError: voiceclonnx not installed` | Re-run Cell 2 (`pip install "ww_trainer[vc]"`). |
| `ValueError: Unknown voiceclonnx engine` | Run Cell 4 — pick a name from `list_engines()`. |
| First conversion hangs / slow | Models are downloading from Hugging Face on first use; let it finish (cached afterwards). |
| Converted clip doesn't say the word | Source clip too short/noisy, or engine struggled — try a cleaner source or a different engine (e.g. `freevc`). |
| All outputs sound identical | Donor folder has too few distinct voices — add more donors. |
| `rvc` errors on a WAV donor | `rvc` is any-to-one: the donor must be a trained RVC `.onnx` model, not a reference WAV. Use `knnvc`/`freevc`/`facodec` for the donor-folder workflow. |